install required pakeages

In [1]:
# Install required packages
!pip -q install datasets pandas numpy scikit-learn

In [2]:
import numpy as np
import pandas as pd

from datasets import load_dataset

from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

Load the dataset

In [3]:
dataset = load_dataset("tattabio/ec_classification")

train_df = pd.DataFrame(dataset["train"])
test_df = pd.DataFrame(dataset["test"])

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_df.head()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/485 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/285k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/87.5k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/512 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/128 [00:00<?, ? examples/s]

Train shape: (512, 3)
Test shape: (128, 3)


,Entry,Label,Sequence
0,Q9LQC0,1.14.14.18,MATSRLNASCRFPASRRLDCESYVSLRAKTVTIRYVRTIAAPRRHL...
1,A0AFT8,1.14.14.18,MIIVTNTIKVEKGAAEHVIRQFTGANGDGHPTKDIAEVEGFLGFEL...
2,P74133,1.14.14.18,MTNLAQKLRYGTQQSHTLAENTAYMKCFLKGIVEREPFRQLLANLY...
3,O31534,1.14.14.18,MFVQLRKMTVKEGFADKVIERFSAEGIIEKQEGLIDVTVLEKNVRR...
4,P34697,1.15.1.1,MFMNLLTQVSNAIFPQVEAAQKMSNRAVAVLRGETVTGTIWITQKS...


In [4]:
print("Number of unique labels in train:", train_df["Label"].nunique())
print("Number of unique labels in test:", test_df["Label"].nunique())

print("\nTop label counts in train:")
print(train_df["Label"].value_counts().head(10))

Number of unique labels in train: 128
Number of unique labels in test: 128

Top label counts in train:
Label
1.14.14.18    4
1.15.1.1      4
1.16.3.1      4
1.17.4.1      4
1.5.98.3      4
1.8.3.2       4
2.1.1.354     4
2.1.1.360     4
2.1.1.37      4
2.1.1.72      4
Name: count, dtype: int64


Define dipeptide features

In [5]:
AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY")
DIPEPTIDES = [a + b for a in AMINO_ACIDS for b in AMINO_ACIDS]
DIPEPTIDE_TO_INDEX = {dp: i for i, dp in enumerate(DIPEPTIDES)}

print("Number of dipeptides:", len(DIPEPTIDES))
print("First 20 dipeptides:", DIPEPTIDES[:20])

Number of dipeptides: 400
First 20 dipeptides: ['AA', 'AC', 'AD', 'AE', 'AF', 'AG', 'AH', 'AI', 'AK', 'AL', 'AM', 'AN', 'AP', 'AQ', 'AR', 'AS', 'AT', 'AV', 'AW', 'AY']


In [6]:
def sequence_to_dipeptide_frequency(seq):
    """
    Convert a protein sequence into a 400-dimensional dipeptide frequency vector.
    """
    seq = str(seq).upper()
    counts = np.zeros(len(DIPEPTIDES), dtype=float)
    valid_count = 0

    for i in range(len(seq) - 1):
        dp = seq[i:i+2]
        if dp[0] in AMINO_ACIDS and dp[1] in AMINO_ACIDS:
            counts[DIPEPTIDE_TO_INDEX[dp]] += 1
            valid_count += 1

    if valid_count > 0:
        counts /= valid_count

    return counts

In [7]:
def build_dipeptide_feature_matrix(sequences):
    """
    Convert a list/series of sequences into a dipeptide feature matrix.
    """
    return np.array([sequence_to_dipeptide_frequency(seq) for seq in sequences])

Build X and y

In [8]:
X_train = build_dipeptide_feature_matrix(train_df["Sequence"])
X_test = build_dipeptide_feature_matrix(test_df["Sequence"])

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (512, 400)
X_test shape: (128, 400)


In [9]:
label_encoder = LabelEncoder()

y_train = label_encoder.fit_transform(train_df["Label"])
y_test = label_encoder.transform(test_df["Label"])

print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)
print("Number of classes:", len(label_encoder.classes_))

y_train shape: (512,)
y_test shape: (128,)
Number of classes: 128


Quick sanity check

In [10]:
example_seq = train_df.loc[0, "Sequence"]
example_vec = sequence_to_dipeptide_frequency(example_seq)

print("Example sequence:")
print(example_seq[:120] + "..." if len(example_seq) > 120 else example_seq)

print("\nFeature vector length:", len(example_vec))
print("Sum of frequencies:", example_vec.sum())
print("Number of nonzero dipeptides:", np.count_nonzero(example_vec))

Example sequence:
MATSRLNASCRFPASRRLDCESYVSLRAKTVTIRYVRTIAAPRRHLVRRANEDQTLVVNVVAAAGEKPERRYPREPNGFVEEMRFVVMKIHPRDQVKEGKSDSNDLVSTWNFTIEGYLKF...

Feature vector length: 400
Sum of frequencies: 0.9999999999999998
Number of nonzero dipeptides: 184


In [11]:
feature_preview = pd.DataFrame(
    X_train[:3, :20],
    columns=[f"dp_{dp}" for dp in DIPEPTIDES[:20]]
)
feature_preview["Label"] = train_df["Label"].iloc[:3].values
feature_preview

,dp_AA,dp_AC,dp_AD,dp_AE,dp_AF,dp_AG,dp_AH,dp_AI,dp_AK,dp_AL,...,dp_AN,dp_AP,dp_AQ,dp_AR,dp_AS,dp_AT,dp_AV,dp_AW,dp_AY,Label
0,0.010638,0.0,0.0,0.014184,0.003546,0.010638,0.003546,0.003546,0.003546,0.000000,...,0.003546,0.003546,0.000000,0.000000,0.007092,0.003546,0.0,0.0,0.007092,1.14.14.18
1,0.008333,0.0,0.0,0.016667,0.000000,0.000000,0.008333,0.000000,0.000000,0.000000,...,0.008333,0.000000,0.008333,0.008333,0.000000,0.000000,0.0,0.0,0.000000,1.14.14.18
2,0.012048,0.0,0.0,0.008032,0.004016,0.004016,0.004016,0.008032,0.004016,0.012048,...,0.008032,0.000000,0.004016,0.004016,0.004016,0.004016,0.0,0.0,0.004016,1.14.14.18


Train the logistic regression model

In [12]:
model = LogisticRegression(
    C=1.0,
    max_iter=3000,
    solver="lbfgs",
    class_weight="balanced"
)

model.fit(X_train, y_train)

LogisticRegression(class_weight='balanced', max_iter=3000)

In [13]:
y_pred = model.predict(X_test)

evaluate the model

In [14]:
accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")
weighted_f1 = f1_score(y_test, y_pred, average="weighted")

print(f"Accuracy: {accuracy:.4f}")
print(f"Macro F1: {macro_f1:.4f}")
print(f"Weighted F1: {weighted_f1:.4f}")

Accuracy: 0.0391
Macro F1: 0.0249
Weighted F1: 0.0249


In [15]:
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

Classification Report:
              precision    recall  f1-score   support

  1.14.14.18       0.00      0.00      0.00         1
    1.15.1.1       0.00      0.00      0.00         1
    1.16.3.1       0.00      0.00      0.00         1
    1.17.4.1       0.00      0.00      0.00         1
    1.5.98.3       0.00      0.00      0.00         1
     1.8.3.2       0.00      0.00      0.00         1
   2.1.1.354       0.00      0.00      0.00         1
   2.1.1.360       0.00      0.00      0.00         1
    2.1.1.37       0.00      0.00      0.00         1
    2.1.1.72       0.00      0.00      0.00         1
    2.1.1.77       0.00      0.00      0.00         1
    2.1.1.86       0.00      0.00      0.00         1
    2.1.3.15       0.00      0.00      0.00         1
   2.3.1.225       0.00      0.00      0.00         1
   2.3.1.269       0.00      0.00      0.00         1
    2.3.1.48       0.00      0.00      0.00         1
    2.3.1.51       0.00      0.00      0.00         1
    

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


compare with baseline

In [16]:
majority_label = train_df["Label"].mode()[0]
baseline_acc = (test_df["Label"] == majority_label).mean()

print("Majority class in training set:", majority_label)
print(f"Majority-class baseline accuracy: {baseline_acc:.4f}")

Majority class in training set: 1.14.14.18
Majority-class baseline accuracy: 0.0078


show prediction

In [17]:
pred_labels = label_encoder.inverse_transform(y_pred)
true_labels = label_encoder.inverse_transform(y_test)

results_df = pd.DataFrame({
    "Entry": test_df["Entry"],
    "True_Label": true_labels,
    "Predicted_Label": pred_labels,
    "Correct": true_labels == pred_labels
})

results_df.head(20)

,Entry,True_Label,Predicted_Label,Correct
0,A0A0H2XEA6,1.14.14.18,3.1.1.4,False
1,Q5AD07,1.15.1.1,3.1.1.4,False
2,O74831,1.16.3.1,3.1.1.4,False
3,O46310,1.17.4.1,3.1.1.4,False
4,Q8PU58,1.5.98.3,7.1.1.9,False
5,Q5UQV6,1.8.3.2,3.1.1.4,False
6,Q8IRW8,2.1.1.354,3.1.1.4,False
7,Q6BTC8,2.1.1.360,3.1.1.4,False
8,P0DOY1,2.1.1.37,3.1.1.4,False
9,Q09956,2.1.1.72,3.1.1.4,False


In [18]:
print("Number correct:", results_df["Correct"].sum())
print("Number incorrect:", (~results_df["Correct"]).sum())

Number correct: 5
Number incorrect: 123


find the best hyperparameters

In [19]:
hyperparameter_results = []

for C in [0.01, 0.1, 1.0, 10.0]:
    for max_iter in [1000, 3000, 5000]:
        clf = LogisticRegression(
            C=C,
            max_iter=max_iter,
            solver="lbfgs",
            class_weight="balanced"
        )
        clf.fit(X_train, y_train)
        preds = clf.predict(X_test)

        hyperparameter_results.append({
            "C": C,
            "max_iter": max_iter,
            "accuracy": accuracy_score(y_test, preds),
            "macro_f1": f1_score(y_test, preds, average="macro"),
            "weighted_f1": f1_score(y_test, preds, average="weighted")
        })

hyperparameter_df = pd.DataFrame(hyperparameter_results)
hyperparameter_df.sort_values(by=["macro_f1", "accuracy"], ascending=False).reset_index(drop=True)

,C,max_iter,accuracy,macro_f1,weighted_f1
0,10.00,1000,0.164062,0.118738,0.118738
1,10.00,3000,0.164062,0.118738,0.118738
2,10.00,5000,0.164062,0.118738,0.118738
3,0.01,1000,0.125000,0.068444,0.068444
4,0.01,3000,0.125000,0.068444,0.068444
5,0.01,5000,0.125000,0.068444,0.068444
6,0.10,1000,0.125000,0.068444,0.068444
7,0.10,3000,0.125000,0.068444,0.068444
8,0.10,5000,0.125000,0.068444,0.068444
9,1.00,1000,0.039062,0.024882,0.024882


In [20]:
best_row = hyperparameter_df.sort_values(
    by=["macro_f1", "accuracy"], ascending=False
).iloc[0]

print(best_row)

C                10.000000
max_iter       1000.000000
accuracy          0.164062
macro_f1          0.118738
weighted_f1       0.118738
Name: 9, dtype: float64


train the model with best hyperparameter

In [21]:
best_model = LogisticRegression(
    C=float(best_row["C"]),
    max_iter=int(best_row["max_iter"]),
    solver="lbfgs",
    class_weight="balanced"
)

best_model.fit(X_train, y_train)
best_pred = best_model.predict(X_test)

print("Best dipeptide model accuracy:", accuracy_score(y_test, best_pred))
print("Best dipeptide model macro F1:", f1_score(y_test, best_pred, average="macro"))
print("Best dipeptide model weighted F1:", f1_score(y_test, best_pred, average="weighted"))

Best dipeptide model accuracy: 0.1640625
Best dipeptide model macro F1: 0.11873759920634921
Best dipeptide model weighted F1: 0.11873759920634921
